In [2]:
import collections, datetime as dt, importlib, json, math, os, random, statistics as st, string, sys, tempfile, warnings
import numpy as np
import pandas as pd

In [3]:
warnings.filterwarnings('ignore')
MONTHS = 'January February March April May June July August September October November December'.split()

In [7]:
def rounding(rng, kind):
    if kind == 'int':
        return rng.choice(['Give a whole number.', 'The answer is an integer.', 'Answer with an integer.']), None
    d = rng.choice([1, 2, 2, 3]); s = 's' * (d > 1)
    return rng.choice([f'Round your answer to {d} decimal place{s}.', f'Round to {d} decimals.',
                       f"Give the answer rounded to {['one', 'two', 'three'][d - 1]} decimal place{s}."]), d


def make(tid, objective, files, gold, d, typ):
    g = float(gold)
    return dict(id=tid, objective=objective, files=files, answer=round(g) if d is None else round(g, d),
                tol=1e-6 if d is None else 1.5 * 10 ** -d, type=typ)


def code(rng, taken):
    while True:
        c = rng.choice(string.ascii_lowercase) + ''.join(rng.choice(string.ascii_lowercase + string.digits) for _ in range(rng.randint(1, 4)))
        c = c.upper() if rng.random() < 0.3 else c
        if c not in taken:
            taken.add(c); return c


def fmt(v, miss, comma):
    if pd.isna(v):
        return miss
    s = str(int(v)) if float(v).is_integer() else repr(round(float(v), 6))
    return s.replace('.', ',') if comma else s


def M(phrase, desc, unit, gen):
    return dict(phrase=phrase, desc=desc, unit=unit, gen=gen)


In [8]:

I = lambda lo, hi: (lambda r, n: r.integers(lo, hi, n).astype(float))
DOMAINS = [
    dict(file='meter.csv', ent='readings', ent1='reading', intro="daily electricity readings of a company's sites",
         id_desc='Reading number', date_desc='Day of the reading', noun='site', cdesc='Site, coded', prefix='SITE-',
         codes='S01 S02 S03 S04 S05'.split(), labels=['Bordeaux office', 'Grenoble lab', 'Lille warehouse', 'Nantes office', 'Toulouse workshop'],
         cat2=('tariff applied', 'Tariff applied', ['peak', 'offpeak', 'flat']),
         meas=[M('electricity used', 'Electricity used over the day', 'kWh', lambda r, n: r.normal(300, 90, n).round(1)),
               M('outside temperature', 'Mean outside temperature', '°C', lambda r, n: r.normal(12, 7, n).round(1)),
               M('people present', 'People present on site', 'people', I(0, 150))],
         look=('sites.csv', 'Site code', [M('floor surface', 'Floor surface', 'square metres', I(800, 6000)),
                                          M('year of construction', 'Year of construction', '', I(1960, 2020))])),
    dict(file='orders.csv', ent='orders', ent1='order', intro='orders of a retail chain', id_desc='Order identifier',
         date_desc='Date of the order', noun='store', cdesc='Store, coded', prefix='STORE-', codes='ST1 ST2 ST3 ST4 ST5 ST6'.split(),
         labels=['Lyon centre', 'Paris Bercy', 'Lille gare', 'Nice port', 'Rennes nord', 'Metz ville'],
         cat2=('sales channel', 'Sales channel', ['web', 'shop', 'phone']),
         meas=[M('order value', 'Value of the order', 'euros', lambda r, n: r.gamma(3, 40, n).round(2)),
               M('number of items', 'Number of items in the order', 'items', I(1, 12)),
               M('discount granted', 'Discount granted on the order', '%', lambda r, n: r.choice([0, 5, 10, 15, 20], n).astype(float))],
         look=('stores.csv', 'Store code', [M('staff count', 'Number of employees', 'people', I(5, 60)),
                                           M('sales area', 'Sales area', 'square metres', I(200, 3000))])),
    dict(file='trips.csv', ent='trips', ent1='trip', intro='freight trips of a logistics company', id_desc='Trip number',
         date_desc='Departure date', noun='carrier', cdesc='Carrier, coded', prefix='CAR-', codes='C01 C02 C03 C04'.split(),
         labels=['Rapidex', 'Nordfret', 'Sudline', 'Alpha Cargo'], cat2=('delivery status', 'Delivery status', ['on time', 'late']),
         meas=[M('distance', 'Distance driven', 'km', lambda r, n: r.normal(420, 160, n).clip(20).round(1)),
               M('duration', 'Duration of the trip', 'hours', lambda r, n: r.normal(6.5, 2.2, n).clip(0.5).round(2)),
               M('load carried', 'Load carried', 'tonnes', lambda r, n: r.normal(14, 5, n).clip(0.5).round(1))],
         look=('carriers.csv', 'Carrier code', [M('fleet size', 'Number of trucks', 'trucks', I(10, 300)),
                                               M('year founded', 'Year the company was founded', '', I(1950, 2015))])),
    dict(file='stays.csv', ent='stays', ent1='stay', intro='hospital stays', id_desc='Stay identifier', date_desc='Admission date',
         noun='ward', cdesc='Ward, coded', prefix='WARD-', codes='W1 W2 W3 W4 W5'.split(),
         labels=['Cardiology', 'Orthopaedics', 'Neurology', 'Oncology', 'Paediatrics'], cat2=('admission type', 'Admission type', ['emergency', 'planned']),
         meas=[M('length of stay', 'Length of stay', 'days', I(1, 25)),
               M('treatment cost', 'Cost of the treatment', 'euros', lambda r, n: r.gamma(4, 900, n).round(2)),
               M('patient age', 'Age of the patient', 'years', I(1, 95))],
         look=('wards.csv', 'Ward code', [M('number of beds', 'Number of beds', 'beds', I(10, 80)), M('floor', 'Floor of the ward', '', I(0, 8))])),
]


In [ ]:
def build(rng, dom):
    nr = np.random.default_rng(rng.randint(0, 10**9)); n = rng.randint(150, 400); taken = set()
    cols = {k: code(rng, taken) for k in ('id', 'cat', 'date', 'cat2')}
    df = pd.DataFrame({'id': nr.permutation(np.arange(1000, 1000 + 3 * n))[:n], 'cat': nr.choice(dom['codes'], n),
                       'date': pd.to_datetime('2022-01-01') + pd.to_timedelta(nr.integers(0, 3 * 365, n), unit='D'),
                       'cat2': nr.choice(dom['cat2'][2], n)})
    meas = []
    for k, m in enumerate(dom['meas']):
        df[f'm{k}'] = m['gen'](nr, n); cols[f'm{k}'] = code(rng, taken); meas.append(dict(m, key=f'm{k}'))
    for m in meas:
        if rng.random() < 0.6:
            df.loc[nr.random(n) < rng.uniform(0.03, 0.12), m['key']] = np.nan
    sentinel = None
    nonneg = [m for m in meas if (df[m['key']].dropna() >= 0).all()]
    if nonneg and rng.random() < 0.3:
        m = rng.choice(nonneg)
        idx = df[df[m['key']].notna()].sample(frac=rng.uniform(0.03, 0.08), random_state=rng.randint(0, 10**6)).index
        df.loc[idx, m['key']] = np.nan; sentinel = (m, idx)
    lfile, ldesc, lms = dom['look']; ltaken = set()
    look = pd.DataFrame({'key': dom['codes']}); lcols = {'key': code(rng, ltaken)}; lmeas = []
    for k, m in enumerate(lms):
        look[f'l{k}'] = m['gen'](nr, len(dom['codes'])); lcols[f'l{k}'] = code(rng, ltaken); lmeas.append(dict(m, key=f'l{k}'))
    return dict(dom=dom, df=df, cols=cols, meas=meas, sentinel=sentinel, look=look, lcols=lcols, lmeas=lmeas)


def write(rng, ds, join):
    dom, df, cols, f = ds['dom'], ds['df'], ds['cols'], ds['dom']['file']
    sep = rng.choice([';', ';', ',']); comma = sep == ';' and rng.random() < 0.8
    miss = rng.choice(['-', '-', 'n/a', 'NA', '?', 'missing', 'unknown', '--'])
    out = df.copy(); sents = [f"{f} uses '{sep}' to separate columns and {'a comma' if comma else 'a point'} as the decimal mark."]
    has_missing = any(df[m['key']].isna().any() for m in ds['meas'])
    if ds['sentinel']:
        sm, idx = ds['sentinel']; out.loc[idx, sm['key']] = -1
        has_missing = any(df[m['key']].drop(idx if m is sm else []).isna().any() for m in ds['meas'])
        sents.append(rng.choice([f"A {sm['phrase']} of -1 means not recorded.", f"The value -1 in the {sm['phrase']} column means not recorded."]))
    if has_missing or ds['sentinel']:
        sents.append((f"Missing values are written '{miss}'. " if has_missing else '') + 'Ignore values that are missing or not recorded.')
    if rng.random() < 0.3:
        out = pd.concat([out, out.sample(frac=rng.uniform(0.03, 0.08), random_state=rng.randint(0, 10**6))]).sample(frac=1, random_state=rng.randint(0, 10**6))
        sents.append(f'{f} contains duplicated rows (same identifier); keep one copy of each.')
    if join and rng.random() < 0.5:
        out['cat'] = dom['prefix'] + out['cat']
        sents.append(f"In {f} the {dom['noun']} key carries the prefix '{dom['prefix']}'; {dom['look'][0]} does not.")
    text = pd.DataFrame({cols['id']: out['id'].astype(int).astype(str), cols['cat']: out['cat'],
                         cols['date']: out['date'].dt.strftime('%Y-%m-%d'), cols['cat2']: out['cat2'],
                         **{cols[m['key']]: [fmt(v, miss, comma) for v in out[m['key']]] for m in ds['meas']}})
    order = list(text.columns); rng.shuffle(order)
    files = {f: text[order].to_csv(sep=sep, index=False)}
    coded = dom['cdesc'] + ' (' + ', '.join(f'{c} {l}' for c, l in zip(dom['codes'], dom['labels'])) + ')'
    dd = [(f, cols['id'], dom['id_desc'], ''), (f, cols['cat'], coded, ''), (f, cols['date'], dom['date_desc'], ''),
          (f, cols['cat2'], f"{dom['cat2'][1]} ({', '.join(dom['cat2'][2])})", '')] + [(f, cols[m['key']], m['desc'], m['unit']) for m in ds['meas']]
    if join:
        lf = dom['look'][0]
        files[lf] = pd.DataFrame({ds['lcols']['key']: ds['look']['key'], **{ds['lcols'][m['key']]: [fmt(v, miss, comma) for v in ds['look'][m['key']]]
                                                                            for m in ds['lmeas']}}).to_csv(sep=sep, index=False)
        sents.append(f'{lf} uses the same separator and decimal mark.')
        dd += [(lf, ds['lcols']['key'], dom['look'][1], '')] + [(lf, ds['lcols'][m['key']], m['desc'], m['unit']) for m in ds['lmeas']]
    files['data_dictionary.csv'] = pd.DataFrame(dd, columns=['file', 'column', 'description', 'unit']).to_csv(index=False)
    return files, sents


AGG = {'mean': ('mean', lambda s: s.mean()), 'median': ('median', lambda s: s.median()), 'maximum': ('maximum', lambda s: s.max()),
       'minimum': ('minimum', lambda s: s.min()), 'total': ('total', lambda s: s.sum()), 'std': ('standard deviation of the', lambda s: s.std())}
CMP = [('at least', lambda s, t: s >= t), ('at most', lambda s, t: s <= t), ('above', lambda s, t: s > t),
       ('below', lambda s, t: s < t), ('more than', lambda s, t: s > t), ('less than', lambda s, t: s < t)]
unit = lambda m: f" (in {m['unit']})" if m['unit'] and m['unit'] != '%' else ''


def thr(rng, s):
    s = s.dropna(); t = float(s.quantile(rng.uniform(0.25, 0.75)))
    return int(round(t)) if (s % 1 == 0).all() or abs(t) > 50 else round(t, 1)


def grp(ds, m, fn):
    return ds['df'].groupby('cat')[m['key']].agg(lambda s: fn(s.dropna()))

def table_question(rng, ds, kind):
    dom, df = ds['dom'], ds['df']; ent = dom['ent']; noun = dom['noun']
    m, m2 = rng.sample(ds['meas'], 2); i = rng.randrange(len(dom['codes'])); c, lab = dom['codes'][i], dom['labels'][i]
    if kind == 'stat':
        w, fn = AGG[rng.choice(list(AGG))]
        return 'table.stat', rng.choice([f"What is the {w} {m['phrase']} over all the {ent}{unit(m)}?", f"What is the {w} {m['phrase']} of all {ent}{unit(m)}?"]), fn(df[m['key']]), 'dec'
    if kind == 'percentile':
        p = rng.choice([10, 25, 75, 90])
        return 'table.percentile', f"What is the {p}th percentile of the {m['phrase']} over all the {ent}{unit(m)}?", df[m['key']].quantile(p / 100), 'dec'
    if kind == 'missing':
        cand = [x for x in ds['meas'] if df[x['key']].isna().any()]
        if not cand: return None
        x = rng.choice(cand)
        return 'table.stat_missing', rng.choice([f"For how many {ent} is the {x['phrase']} missing?", f"How many {ent} have no recorded {x['phrase']}?"]), df[x['key']].isna().sum(), 'int'
    if kind == 'filter':
        w, fn = AGG[rng.choice(['mean', 'median', 'maximum', 'minimum', 'total'])]
        q = rng.choice([f"What is the {w} {m['phrase']} of the {ent} of the {lab} {noun}{unit(m)}?", f"Among the {ent} of the {lab} {noun}, what is the {w} {m['phrase']}{unit(m)}?"])
        return 'table.filter', q, fn(df[df['cat'] == c][m['key']]), 'dec'
    if kind == 'filter_num':
        w, fn = AGG[rng.choice(['mean', 'median', 'total', 'maximum'])]; t = thr(rng, df[m2['key']]); cw, cf = rng.choice(CMP)
        return 'table.filter_num', f"What is the {w} {m['phrase']} of the {ent} with a {m2['phrase']} {'of ' * cw.startswith('at')}{cw} {t}{unit(m)}?", fn(df[cf(df[m2['key']], t)][m['key']]), 'dec'
    if kind == 'count':
        v = rng.choice(dom['cat2'][2]); ph = dom['cat2'][0]
        q = rng.choice([f"How many {ent} have a {ph} of '{v}'?", f"How many {ent} are there whose {ph} is '{v}'?", f"How many {ent} with '{v}' as {ph} are there?"])
        return 'table.count', q, (df['cat2'] == v).sum(), 'int'
    if kind == 'count_num':
        t = thr(rng, df[m['key']]); cw, cf = rng.choice(CMP)
        return 'table.count_num', rng.choice([f"How many {ent} have a {m['phrase']} {cw} {t}?", f"How many {ent} are there with a {m['phrase']} {cw} {t}?"]), cf(df[m['key']], t).sum(), 'int'
    if kind == 'share_within':
        v = rng.choice(dom['cat2'][2]); sub = df[df['cat'] == c]; pct = rng.random() < 0.6
        if sub.empty: return None
        what = 'what percentage' if pct else 'what fraction (between 0 and 1)'
        return 'table.share_within', f"Among the {ent} of the {lab} {noun}, {what} have a {dom['cat2'][0]} of '{v}'?", (sub['cat2'] == v).mean() * (100 if pct else 1), 'dec'
    if kind in ('group', 'rank', 'second'):
        a = rng.choice(['mean', 'total']); w, fn = AGG[a]; g = grp(ds, m, fn)
        if kind == 'group':
            big = rng.random() < 0.5; lg = 'largest' if big else 'smallest'
            q = rng.choice([f"Group the {ent} by the {noun} and compute the {w} {m['phrase']} per group. What is the {lg} group {a}?",
                            f"Compute the {w} {m['phrase']} for each {noun}. What is the {lg} value?"])
            return 'table.group', q, g.max() if big else g.min(), 'dec'
        if kind == 'rank':
            return 'table.rank', f"Rank the {noun} values by the {w} {m['phrase']}, with rank 1 for the largest. What is the rank of {rng.choice([c, f'the {lab} {noun}'])}?", g.rank(ascending=False, method='min')[c], 'int'
        big = rng.random() < 0.6; g = g.sort_values()
        return 'table.second', f"What is the second-{'largest' if big else 'smallest'} {w} {m['phrase']} among the {noun}s{unit(m)}?", g.iloc[-2] if big else g.iloc[1], 'dec'
    if kind == 'spread':
        g = df.groupby('cat')[m['key']].mean()
        return 'table.spread', f"What is the difference between the largest and the smallest mean {m['phrase']} among the {noun}s{unit(m)}?", g.max() - g.min(), 'dec'
    if kind == 'difference':
        (c1, l1), (c2, l2) = rng.sample(list(zip(dom['codes'], dom['labels'])), 2); g = df.groupby('cat')[m['key']].mean()
        return 'table.difference', f"What is the mean {m['phrase']} of the {l1} {noun} minus the mean {m['phrase']} of the {l2} {noun}{unit(m)}?", g[c1] - g[c2], 'dec'
    if kind == 'growth':
        y = df['date'].dt.year; y1 = rng.choice([2022, 2023]); a, b = df[y == y1][m['key']].sum(), df[y == y1 + 1][m['key']].sum()
        return None if a == 0 else ('table.growth', f"What is the percentage change in the total {m['phrase']} from {y1} to {y1 + 1}?", 100 * (b / a - 1), 'dec')
    if kind == 'date_year':
        y = rng.choice([2022, 2023, 2024]); w, fn = AGG[rng.choice(['mean', 'total', 'median'])]
        q = rng.choice([f"What is the {w} {m['phrase']} of the {ent} in {y}{unit(m)}?", f"What is the {w} {m['phrase']} of the {ent} dated in {y}{unit(m)}?"])
        return 'table.date_year', q, fn(df[df['date'].dt.year == y][m['key']]), 'dec'
    if kind == 'date_after':
        d = pd.Timestamp('2022-01-01') + pd.Timedelta(days=rng.randint(200, 900))
        return 'table.date_after', f"What is the mean {m['phrase']} of the {ent} after {d.day} {MONTHS[d.month - 1]} {d.year}{unit(m)}?", df[df['date'] > d][m['key']].mean(), 'dec'
    if kind == 'distinct':
        t = thr(rng, df[m['key']])
        return 'table.distinct', f"How many distinct {noun}s have at least one {dom['ent1']} with a {m['phrase']} above {t}?", df[df[m['key']] > t]['cat'].nunique(), 'int'
    ok = df[m['key']].notna() & df[m2['key']].notna()
    if kind == 'weighted':
        if df.loc[ok, m2['key']].sum() == 0: return None
        return 'table.weighted', f"What is the weighted mean of the {m['phrase']} weighted by the {m2['phrase']} over all the {ent}?", (df.loc[ok, m['key']] * df.loc[ok, m2['key']]).sum() / df.loc[ok, m2['key']].sum(), 'dec'
    if kind == 'ratio_totals':
        if df.loc[ok, m2['key']].sum() == 0: return None
        return 'table.ratio_totals', f"What is the total {m['phrase']} divided by the total {m2['phrase']}, using only {ent} where both are recorded?", df.loc[ok, m['key']].sum() / df.loc[ok, m2['key']].sum(), 'dec'
    if kind == 'correlation':
        return 'table.correlation', f"What is the correlation between the {m['phrase']} and the {m2['phrase']} over all the {ent}?", df[m['key']].corr(df[m2['key']]), 'dec'
    s = df[m['key']]
    if kind == 'iqr':
        return 'table.iqr', f"What is the interquartile range of the {m['phrase']} over all the {ent}{unit(m)}?", s.quantile(.75) - s.quantile(.25), 'dec'
    if kind == 'range':
        return 'table.range', f"What is the range (maximum minus minimum) of the {m['phrase']} over all the {ent}{unit(m)}?", s.max() - s.min(), 'dec'
    if kind == 'derive':
        r = (s / df[m2['key']].where(df[m2['key']] != 0)).replace([np.inf, -np.inf], np.nan); a = rng.choice(['mean', 'median'])
        if rng.random() < 0.5:
            r, tail = r[df['cat'] == c], f" over the {ent} of the {lab} {noun}"
        else:
            tail = f" over all the {ent}"
        q = f"Consider the {m['phrase']} per {m2['phrase']} (the {m['phrase']} divided by the {m2['phrase']}). What is its {a}{tail}?"
        return 'table.derive', q, r.mean() if a == 'mean' else r.median(), 'dec'
    if kind == 'join':
        lm = rng.choice(ds['lmeas']); t = thr(rng, ds['look'][lm['key']]); cw, cf = rng.choice(CMP[:4])
        sub = df[df['cat'].isin(ds['look'].loc[cf(ds['look'][lm['key']], t), 'key'])][m['key']]
        if sub.dropna().empty: return None
        w, fn = AGG[rng.choice(['mean', 'median', 'total'])]; lu = f" {lm['unit']}" if lm['unit'] else ''
        return 'table.join', f"What is the {w} {m['phrase']} of the {ent} that belong to a {noun} with a {lm['phrase']} {'of ' * cw.startswith('at')}{cw} {t}{lu}{unit(m)}?", fn(sub), 'dec'


TABLE_KINDS = ('stat stat percentile missing filter filter filter_num count count_num share_within group rank second spread difference '
               'growth date_year date_after distinct weighted correlation iqr range derive derive ratio_totals join join').split()
DISTRACTORS = ['The {l} {n} was renovated in 2021.', 'Management reviews these figures every quarter.', 'The {l} {n} opened a new entrance last spring.']


def table_task(rng, tid):
    while True:
        dom = rng.choice(DOMAINS); ds = build(rng, dom); kind = rng.choice(TABLE_KINDS)
        res = table_question(rng, ds, kind)
        if res is None or res[2] is None or not np.isfinite(float(res[2])):
            continue
        typ, q, gold, k = res; join = kind == 'join'
        files, sents = write(rng, ds, join)
        intro = f"You are given {dom['intro']} in {dom['file']}" + (f" and {dom['look'][0]}" if join else '') + ', with a data dictionary.'
        extra = [rng.choice(DISTRACTORS).format(l=rng.choice(dom['labels']), n=dom['noun'])] if rng.random() < 0.25 else []
        rs, d = rounding(rng, k)
        return make(tid, ' '.join([intro] + extra + [q] + sents + [rs]), files, gold, d, typ)


############ CALC #############################################
def binom(n, p, k):
    return math.comb(n, k) * p ** k * (1 - p) ** (n - k)


def calc_task(rng, tid):
    R = rng.randint; C = rng.choice
    kind = C('arithmetic stats_inline percent growth functions dates integer word probability'.split())
    if kind == 'arithmetic':
        a, b, c, d = (R(2, 99) for _ in range(4))
        forms = [(f'Compute {a} × {b}.', a * b), (f'What is {a} + {b} × {c}?', a + b * c), (f'Compute ({a} + {b}) × {c} - {d}.', (a + b) * c - d),
                 (f'What is {a * b} divided by {b}?', a), (f'Compute {a}² - {b}.', a * a - b), (f'What is {a} × {b} - {c} × {d}?', a * b - c * d)]
        k = 'int'
    elif kind == 'stats_inline':
        v = [R(1, 60) for _ in range(R(5, 9))]; lst = ', '.join(map(str, v))
        name, val = C([('mean', st.mean(v)), ('median', st.median(v)), ('sample standard deviation', st.stdev(v)),
                       ('population standard deviation', st.pstdev(v)), ('sample variance', st.variance(v)), ('range', max(v) - min(v))])
        if rng.random() < 0.2:
            t = R(10, 50); forms, k = [(f'How many of the values {lst} are strictly above {t}?', sum(x > t for x in v))], 'int'
        else:
            forms, k = [(C([f'The values are {lst}. What is the {name}?', f'What is the {name} of {lst}?', f'A sample contains {lst}. Compute its {name}.']), val)], 'dec'
    elif kind == 'percent':
        p = C([5, 8, 10, 12, 15, 20, 25, 30, 40]); x = R(20, 899); cur = C(['euros', 'dollars'])
        forms = [(f'What is {p}% of {x}?', x * p / 100), (f'A price of {x} {cur} is reduced by {p}%. What is the new price?', x * (1 - p / 100)),
                 (f'A price of {x} {cur} is increased by {p}%. What is the new price?', x * (1 + p / 100)),
                 (f'After a {p}% discount, a jacket costs {x} {cur}. What was the original price?', x / (1 - p / 100)),
                 (f'A bill of {x} {cur} includes {p}% VAT. What is the amount before VAT?', x / (1 + p / 100)),
                 (f'A salary rises from {x} to {x + 3 * p} {cur}. By what percentage did it increase?', 100 * 3 * p / x),
                 (f'A price rises by {p}% and then falls by {p}%. What is the overall change in percent?', 100 * ((1 + p / 100) * (1 - p / 100) - 1))]
        k = 'dec'
    elif kind == 'growth':
        p = C([2, 3, 4, 5, 6, 8, 10]); x = rng.randrange(500, 20000, 50); n = R(2, 12); xs = f'{x:,}'; y = round(x * (1 + p / 100) ** n)
        forms = [(f'An account holds {xs} euros with {p}% compound interest per year. What is the balance after {n} years?', x * (1 + p / 100) ** n),
                 (f'A machine worth {xs} euros loses {p}% of its value every year. What is its value after {n} years?', x * (1 - p / 100) ** n),
                 (f'A car worth {xs} euros depreciates by {p}% per year. What is it worth after {n} years?', x * (1 - p / 100) ** n),
                 (f'A population grew from {x} to {y} in {n} years. What is the constant annual growth rate in percent?', 100 * ((y / x) ** (1 / n) - 1))]
        k = 'dec'
        if rng.random() < 0.2:
            forms, k = [(f'How many full years are needed for {xs} euros at {p}% compound interest per year to reach at least {2 * x:,} euros?', math.ceil(math.log(2) / math.log(1 + p / 100)))], 'int'
    elif kind == 'functions':
        x = R(2, 5000); e = round(x / 1000, 2); b, pw = R(2, 9), R(2, 6)
        forms = [(f'What is the square root of {x}?', math.sqrt(x)), (f'What is the base-10 logarithm of {x}?', math.log10(x)),
                 (f'What is the natural logarithm of {x}?', math.log(x)), (f'What is the base-2 logarithm of {x}?', math.log2(x)),
                 (f'What is e raised to the power {e}?', math.exp(e)), (f'What is {b} raised to the power {pw}?', b ** pw), (f'What is the cube root of {x}?', x ** (1 / 3))]
        k = 'dec'
    elif kind == 'dates':
        d1 = dt.date(2015, 1, 1) + dt.timedelta(days=R(0, 3600)); n = R(3, 400); d2 = d1 + dt.timedelta(days=n)
        D = lambda d: C([f'{d.day} {MONTHS[d.month - 1]} {d.year}', f'{MONTHS[d.month - 1]} {d.day}, {d.year}', d.isoformat()])
        forms = [(f'How many days are there from {D(d1)} to {D(d2)}?', n), (f'How many days are there between {D(d1)} and {D(d2)}?', n),
                 (f'How many days are there from {D(d1)} to {D(d2)}, counting both the first and the last day?', n + 1),
                 (f'What is the date {n} days after {D(d1)}, written as YYYYMMDD?', int(d2.strftime('%Y%m%d'))),
                 (f'On which day of the week does {D(d1)} fall? Monday is 1 and Sunday is 7.', d1.isoweekday())]
        k = 'int'
    elif kind == 'integer':
        a, b, n, r = R(6, 120), R(6, 120), R(5, 15), R(2, 4); p, q = a % 30 + 5, b % 30 + 5
        forms = [(f'What is the greatest common divisor of {a} and {b}?', math.gcd(a, b)), (f'What is the least common multiple of {a} and {b}?', math.lcm(a, b)),
                 (f'What is the remainder of the division of {7 * a + 3} by {b}?', (7 * a + 3) % b), (f'In how many ways can {r} people be chosen from {n}?', math.comb(n, r)),
                 (f'How many ordered arrangements of {r} items chosen from {n} are there?', math.perm(n, r)),
                 (f'Two buses leave together, one every {p} minutes and one every {q} minutes. After how many minutes do they leave together again?', math.lcm(p, q))]
        k = 'int'
    elif kind == 'word':
        f, e, x, y, v1, v2, km, w, h, r = R(2, 9), R(10, 30), R(1, 20), R(1, 20), R(40, 90), R(40, 90), R(60, 400), R(3, 20), R(20, 40), R(12, 30)
        forms = [(f'A pump fills a tank in {f} hours and a drain empties it in {e} hours. How many hours does it take to fill the tank with both working?', 1 / (1 / f - 1 / e), 'dec'),
                 (f'Solve the system 2x + y = {2 * x + y} and x - y = {x - y}. What is the value of x?', x, 'dec'),
                 (f'A car drives {km} km at {v1} km/h and comes back at {v2} km/h. What is its average speed for the whole trip in km/h?', 2 * km / (km / v1 + km / v2), 'dec'),
                 (f'A firm employs {w} people, each working {h} hours a week at {r} euros an hour. What is the total paid over 4 weeks?', 4 * w * h * r, 'dec'),
                 (f'One foot is 12 inches. How many inches are {x} feet and {y % 12} inches?', 12 * x + y % 12, 'int'),
                 (f'A train travels {km} km in {round(km / v1, 2)} hours. What is its average speed in km/h?', km / round(km / v1, 2), 'dec'),
                 (f'Anna is {x} years older than Ben. In {y} years the sum of their ages will be {2 * w + x + 2 * y}. How old is Ben now?', w, 'int')]
        q, val, k = C(forms); forms = [(q, val)]
    else:
        p = C([0.05, 0.1, 0.15, 0.2, 0.3]); n = R(5, 20); j = R(1, 3); pct = rng.random() < 0.3; sc = 100 if pct else 1
        ask = 'what is the probability, in percent,' if pct else 'what is the probability'; Ask = ask[0].upper() + ask[1:]
        prev, sens, fpr = C([1, 2, 5]), C([90, 95, 99]), C([2, 3, 5, 10]); g1, p1, g2 = R(5, 50), C([20, 30, 40]), R(2, 20)
        s = min(R(4, 15), 12); red, blue = R(3, 8), R(3, 8); part = f'Each part has probability {p} of being defective. Among {n} parts, {ask} of'
        forms = [(f'{part} at least {j} defective parts?', sc * (1 - sum(binom(n, p, i) for i in range(j)))),
                 (f'{part} exactly {j} defective parts?', sc * binom(n, p, j)), (f'{part} at most {j} defective parts?', sc * sum(binom(n, p, i) for i in range(j + 1))),
                 (f'Two fair six-sided dice are rolled. {Ask} that they add up to {s}?', sc * sum(a + b == s for a in range(1, 7) for b in range(1, 7)) / 36),
                 (f'A disease has prevalence {prev}%. A test has sensitivity {sens}% and a false positive rate of {fpr}%. What is the probability of disease given a positive test, in percent?',
                  100 * prev * sens / (prev * sens + (100 - prev) * fpr)),
                 (f'A game pays {g1} euros with probability {p1}% and costs {g2} euros otherwise. What is the expected gain in euros?', g1 * p1 / 100 - g2 * (1 - p1 / 100)),
                 (f'An urn holds {red} red and {blue} blue balls. Two balls are drawn without replacement. {Ask} that both are red?', sc * red * (red - 1) / ((red + blue) * (red + blue - 1)))]
        k = 'dec'
    q, val = C(forms); rs, d = rounding(rng, k)
    return make(tid, q + ' ' + rs, {}, val, d, f'calc.{kind}')


######################## FIT ############################################################
FIT_DOMAINS = [
    dict(kind='logistic', intro='consumer loans of a bank', ent='loans', ent1='loan', id_desc='Loan identifier', target_desc='Loan defaulted (1) or repaid (0)',
         target='default (1) versus repaid (0)', pos='default', leak='File passed to a collection agency (1/0)', year=('year granted', 'Year the loan was granted'),
         feats=[M('income', 'Annual income', 'euros', lambda r, n: r.normal(40000, 12000, n).round(0)),
                M('loan amount', 'Loan amount', 'euros', lambda r, n: r.normal(15000, 5000, n).clip(1000).round(0)),
                M('loan duration', 'Loan duration', 'months', I(6, 84)), M('age', 'Age of the borrower', 'years', I(19, 75))], beta=[-1.0, -0.00003, 0.00004, 0.015, -0.01]),
    dict(kind='logistic', intro='customers of a phone operator', ent='customers', ent1='customer', id_desc='Customer identifier', target_desc='Customer left (1) or stayed (0)',
         target='churn (1) versus stay (0)', pos='churn', leak='Cancellation letter received (1/0)', year=('signup year', 'Year of signup'),
         feats=[M('monthly fee', 'Monthly fee', 'euros', lambda r, n: r.normal(35, 12, n).clip(5).round(2)), M('tenure', 'Months as a customer', 'months', I(1, 120)),
                M('support calls', 'Number of calls to support', 'calls', lambda r, n: r.poisson(2, n).astype(float)),
                M('data usage', 'Monthly data usage', 'GB', lambda r, n: r.gamma(2, 6, n).round(1))], beta=[-0.8, 0.02, -0.02, 0.35, -0.02]),
    dict(kind='linear', intro='houses sold in a city', ent='houses', ent1='house', id_desc='House identifier', target_desc='Sale price', target='sale price',
         target_unit='thousand euros', leak='Agency fee charged at sale (euros)', year=('year of sale', 'Year of sale'), noise=25,
         feats=[M('surface', 'Living surface', 'square metres', lambda r, n: r.normal(95, 30, n).clip(20).round(1)), M('number of rooms', 'Number of rooms', 'rooms', I(1, 8)),
                M('distance to the centre', 'Distance to the city centre', 'km', lambda r, n: r.gamma(2, 3, n).round(2)), M('building age', 'Age of the building', 'years', I(0, 120))],
         beta=[60, 2.6, 8, -6, -0.4]),
    dict(kind='linear', intro='office buildings and their energy use', ent='buildings', ent1='building', id_desc='Building identifier', target_desc='Annual energy consumption',
         target='annual energy consumption', target_unit='MWh', leak='Energy bill paid (euros)', year=('year of measurement', 'Year of the measurement'), noise=40,
         feats=[M('floor area', 'Floor area', 'square metres', lambda r, n: r.normal(3000, 900, n).clip(300).round(0)), M('occupants', 'Number of occupants', 'people', I(10, 400)),
                M('mean outside temperature', 'Mean outside temperature', '°C', lambda r, n: r.normal(11, 4, n).round(1)),
                M('window share', 'Share of the facade made of windows', '%', lambda r, n: r.uniform(10, 70, n).round(1))], beta=[120, 0.09, 0.4, -6, 0.8]),
]


def logistic_mle(X, y, intercept=True):
    A = np.column_stack([np.ones(len(X)), X]) if intercept else X
    mu, sd = A.mean(0), A.std(0); sd[sd == 0] = 1
    if intercept: mu[0], sd[0] = 0, 1
    else: mu[:] = 0
    Z = (A - mu) / sd; b = np.zeros(Z.shape[1])
    for _ in range(100):
        p = 1 / (1 + np.exp(-Z @ b)); step = np.linalg.solve(Z.T @ (Z * (p * (1 - p))[:, None]), Z.T @ (y - p)); b += step
        if np.abs(step).max() < 1e-12: break
    beta = b / sd
    if intercept: beta[0] = b[0] - (b[1:] * mu[1:] / sd[1:]).sum()
    return beta


def fit_task(rng, tid):
    while True:
        try:
            return _fit_task(rng, tid)
        except (ValueError, np.linalg.LinAlgError):
            continue


def _fit_task(rng, tid):
    dom = rng.choice(FIT_DOMAINS); nr = np.random.default_rng(rng.randint(0, 10**9)); n = rng.randint(300, 800); feats = dom['feats']; lg = dom['kind'] == 'logistic'
    X = np.column_stack([f['gen'](nr, n) for f in feats]); lin = dom['beta'][0] + X @ np.array(dom['beta'][1:])
    y = (nr.random(n) < 1 / (1 + np.exp(-lin))).astype(float) if lg else (lin + nr.normal(0, dom['noise'], n)).round(1)
    df = pd.DataFrame(X, columns=[f'f{i}' for i in range(len(feats))]); df.insert(0, 'id', nr.permutation(np.arange(1000, 1000 + 3 * n))[:n])
    df['year'] = nr.integers(2015, 2024, n); df['leak'] = ((y == 1) & (nr.random(n) < 0.7)).astype(float) if lg else (y * 30 + nr.normal(0, 50, n)).round(0); df['y'] = y
    miss = rng.choice(['-', 'NA', 'n/a', '?', 'missing']); has_missing = rng.random() < 0.5
    if has_missing:
        for i in range(len(feats)): df.loc[nr.random(n) < 0.03, f'f{i}'] = np.nan
    sentinel = sidx = None
    if rng.random() < 0.3:
        k = rng.randrange(len(feats))
        if (df[f'f{k}'].dropna() >= 0).all():
            sentinel, sidx = k, df.sample(frac=0.05, random_state=rng.randint(0, 10**6)).index
    split = int(n * 0.75); train, test = df.iloc[:split], df.iloc[split:]
    used = sorted(rng.sample(range(len(feats)), rng.randint(2, 3))); intercept = rng.random() < 0.85; conds, ctxt = [], []
    if rng.random() < 0.6:
        yr, later = rng.randint(2017, 2021), rng.random() < 0.5
        conds.append(lambda d, yr=yr, later=later: d['year'] >= yr if later else d['year'] <= yr); ctxt.append(f"with a {dom['year'][0]} of {yr} or {'later' if later else 'earlier'}")
    if rng.random() < 0.3:
        k = rng.choice([i for i in range(len(feats)) if i not in used] or used); t = float(np.nanquantile(train[f'f{k}'], 0.3))
        t = round(t) if abs(t) > 20 else round(t, 1); t = int(t) if float(t).is_integer() else t
        conds.append(lambda d, k=k, t=t: d[f'f{k}'] > t)
        ctxt.append(f"{'' if ctxt else 'with '}{'an' if feats[k]['phrase'][0] in 'aeiou' else 'a'} {feats[k]['phrase']} above {t}")
    tt, te = train.copy(), test.copy()
    if sentinel is not None:
        for t_ in (tt, te): t_.loc[t_.index.intersection(sidx), f'f{sentinel}'] = np.nan
    cols = [f'f{i}' for i in used]
    for c in conds: tt = tt[c(tt)]
    tt = tt.dropna(subset=cols + ['y'])
    if len(tt) < 40 or (lg and tt['y'].nunique() < 2): raise ValueError
    Xt, yt = tt[cols].to_numpy(float), tt['y'].to_numpy(float)
    beta = logistic_mle(Xt, yt, intercept) if lg else np.linalg.lstsq(np.column_stack([np.ones(len(Xt)), Xt]) if intercept else Xt, yt, rcond=None)[0]
    b0, bf = (beta[0], beta[1:]) if intercept else (0.0, beta)
    pred = lambda A: (1 / (1 + np.exp(-(b0 + A @ bf)))) if lg else b0 + A @ bf
    names = [feats[i]['phrase'] for i in used]; tc = te.dropna(subset=cols)
    what = f"predicted probability of {dom['pos']}" if lg else f"predicted {dom['target']}"
    ask = rng.choice(['predict', 'predict', 'coef', 'mean'] + ['intercept'] * intercept + (['count'] if lg else ['r2']))
    if ask == 'predict':
        row = tc.sample(1, random_state=rng.randint(0, 10**6)).iloc[0]
        q, gold = f"What is the {what} for the {dom['ent1']} {int(row['id'])} of test.csv?", pred(row[cols].to_numpy(float)[None, :])[0]
    elif ask == 'coef':
        big = [i for i in range(len(used)) if abs(bf[i]) >= 0.01]
        if not big: raise ValueError
        j = rng.choice(big); q, gold = f'What is the coefficient of the {names[j]}?', bf[j]
    elif ask == 'intercept':
        q, gold = 'What is the intercept?', b0
    elif ask == 'r2':
        q, gold = 'What is the R² of the fit on the training rows?', 1 - ((yt - pred(Xt)) ** 2).sum() / ((yt - yt.mean()) ** 2).sum()
    elif ask == 'count':
        q, gold = f"How many {dom['ent']} of test.csv have a predicted probability of {dom['pos']} above 0.5?", (pred(tc[cols].to_numpy(float)) > 0.5).sum()
    else:
        q, gold = f"What is the mean {what} over the {dom['ent']} of test.csv?", pred(tc[cols].to_numpy(float)).mean()
    taken = set(); cmap = {k: code(rng, taken) for k in ['id', 'year', 'leak', 'y'] + [f'f{i}' for i in range(len(feats))]}
    sep = rng.choice([',', ',', ';']); comma = sep == ';' and rng.random() < 0.7

    def text(d, with_y):
        d = d.copy()
        if sentinel is not None: d.loc[d.index.intersection(sidx), f'f{sentinel}'] = -1
        keys = ['id'] + [f'f{i}' for i in range(len(feats))] + ['year', 'leak'] + ['y'] * with_y
        out = pd.DataFrame({cmap[k]: [fmt(v, miss, comma) for v in d[k]] for k in keys}); order = list(out.columns); rng.shuffle(order)
        return out[order]
    tr = text(train, True); sents = []
    if rng.random() < 0.3:
        tr = pd.concat([tr, tr.sample(frac=0.05, random_state=rng.randint(0, 10**6))]).sample(frac=1, random_state=1)
        sents.append('train.csv contains duplicated rows (same identifier); keep one copy of each.')
    files = {'train.csv': tr.to_csv(sep=sep, index=False), 'test.csv': text(test, False).to_csv(sep=sep, index=False)}
    dd = [(fn, cmap[k], desc, u) for fn, wy in (('train.csv', True), ('test.csv', False))
          for k, desc, u in [('id', dom['id_desc'], '')] + [(f'f{i}', f['desc'], f['unit']) for i, f in enumerate(feats)]
          + [('year', dom['year'][1], ''), ('leak', dom['leak'], '')] + [('y', dom['target_desc'], dom.get('target_unit', ''))] * wy]
    files['data_dictionary.csv'] = pd.DataFrame(dd, columns=['file', 'column', 'description', 'unit']).to_csv(index=False)
    if has_missing: sents.append(f"Missing values are written '{miss}'; drop rows with a missing value in a column you use.")
    if sentinel is not None: sents.append(f"A {feats[sentinel]['phrase']} of -1 means not recorded.")
    model = ('a logistic regression without regularisation, ' if lg else rng.choice(['a linear regression', 'an ordinary least-squares linear regression']) + ' ') \
        + ('with an intercept' if intercept else 'without an intercept')
    flist = ', '.join(names[:-1]) + ' and ' + names[-1]
    sentence = f"Fit {model}, of {'' if lg else 'the '}{dom['target']} on the {flist}, using the {dom['ent']} of train.csv{(' ' + ' and '.join(ctxt)) if ctxt else ''}."
    rs, d = rounding(rng, 'int' if ask == 'count' else 'dec')
    if d is not None and lg and ask in ('coef', 'intercept'):
        d = max(d, 3); rs = f'Round to {d} decimals.'
    obj = ' '.join([f"train.csv and test.csv describe {dom['intro']}; data_dictionary.csv describes the columns.", sentence,
                    f'Do not use the column described as "{dom["leak"]}": it is recorded after the outcome.',
                    f"Both files use '{sep}' to separate columns and {'a comma' if comma else 'a point'} as the decimal mark."] + sents + [q, rs])
    return make(tid, obj, files, gold, d, f"fit.{dom['kind']}")


def generate(n=1500, seed=0, only=None):
    # depending on the seed (different seed will give different tasks)
    rng = random.Random(seed); makers = {'calc': calc_task, 'table': table_task, 'fit': fit_task}
    fams = [only] if only else ['calc', 'table', 'fit']; w = {'calc': 45, 'table': 50, 'fit': 25}
    out = []
    for i in range(n):
        fam = rng.choices(fams, weights=[w[f] for f in fams])[0]; out.append(makers[fam](rng, f'g_{fam[0]}{i:05d}'))
    return out


def save(tasks, path='gen.json'):
    json.dump(tasks, open(path, 'w', encoding='utf-8'), ensure_ascii=False); print(f'{len(tasks)} tasks -> {path}')


In [ ]:
tasks = generate(1500, seed=1)      
save(tasks, 'other_data.json')

1500 tasks -> other_Data.json
